Este código es para calcular los datos de los CSVs de la INEGI

Se probó el simulador.


In [4]:
import pandas as pd
import numpy as np
import json

# ==========================================
#             Rutas de archivos
# ==========================================
RUTA_CENSO = 'conjunto_de_datos_iter_09CSV20.csv'
RUTA_DENUE = 'denue_inegi_09_.csv'

# ==========================================
#                 Censo
# ==========================================
try:
    df_censo_raw = pd.read_csv(RUTA_CENSO, encoding='utf-8-sig', low_memory=False)
except UnicodeDecodeError:
    df_censo_raw = pd.read_csv(RUTA_CENSO, encoding='latin-1', low_memory=False)

# Normalizar nombres de columnas
df_censo_raw.columns = df_censo_raw.columns.str.strip().str.replace('ï»¿', '').str.upper()

print("Columnas del CSV del Censo:", list(df_censo_raw.columns[:12]))

# Filtrar
df_censo = df_censo_raw[df_censo_raw['NOM_LOC'].str.contains('Total del Municipio|Total de la entidad', case=False, na=False) == False].copy()
nivel_censo = 'LOCALIDAD/ITER'

# Columnas clave para caso de estudio
cols_demanda = [
    'POBTOT',     # Población total
    'P_0A2',      # Niñas y niños de 0 a 2 años
    'P_3A5',      # Niñas y niños de 3 a 5 años
    'P_60YMAS',   # Personas adultas mayores (60 años y más)
    'PCON_DISC',  # Personas con discapacidad
    'PEA_F',      # Mujeres económicamente activas
    'PE_INAC_F',  # Mujeres no económicamente activas
    'HOGJEF_F'    # Hogares con jefatura femenina
]

for col in cols_demanda:
    if col in df_censo.columns:
        df_censo[col] = pd.to_numeric(
            df_censo[col].astype(str).str.replace('*', '0', regex=False).str.replace('N/D', '0', regex=False),
            errors='coerce'
        ).fillna(0)

# Crear llaves geográficas compatibles con el json de alcaldías (no se utiliza en este código)
df_censo['CVE_ENT'] = df_censo['ENTIDAD'].astype(str).str.zfill(2)
df_censo['CVE_MUN'] = df_censo['MUN'].astype(str).str.zfill(3)
df_censo['CVEGEO'] = df_censo['CVE_ENT'] + df_censo['CVE_MUN']

# Calcular Demanda Potencial de Cuidados
df_censo['DEMANDA_CUIDADO_TOTAL'] = (
    df_censo.get('P_0A2', 0) +
    df_censo.get('P_3A5', 0) +
    df_censo.get('P_60YMAS', 0) +
    df_censo.get('PCON_DISC', 0)
)

print(f"\nNivel detectado: {nivel_censo} | Filas limpias: {len(df_censo)}")
print(df_censo[['CVEGEO', 'NOM_MUN', 'POBTOT', 'DEMANDA_CUIDADO_TOTAL', 'PE_INAC_F']].head())
print("-" * 60)

# ==========================================
#               Denue
# ==========================================
try:
    df_denue_raw = pd.read_csv(RUTA_DENUE, encoding='latin-1', low_memory=False)
except UnicodeDecodeError:
    df_denue_raw = pd.read_csv(RUTA_DENUE, encoding='utf-8-sig', low_memory=False)

# Normalizar columnas del DENUE a minúsculas sin espacios
df_denue_raw.columns = df_denue_raw.columns.str.strip().str.lower()
print("Primeras 10 columnas del DENUE:", list(df_denue_raw.columns[:10]))

df_denue_raw['codigo_act'] = df_denue_raw['codigo_act'].astype(str).str.strip()

codigos_cuidado = {
    '624411': 'Guardería Privada',
    '624412': 'Guardería Pública',
    '623311': 'Asilo / Residencia Adultos Mayores (Privado)',
    '623312': 'Asilo / Residencia Adultos Mayores (Público)',
    '624121': 'Centro de Día Adultos Mayores / Discapacidad (Privado)',
    '624122': 'Centro de Día Adultos Mayores / Discapacidad (Público)',
    '611111': 'Educación Preescolar Privada',
    '611112': 'Educación Preescolar Pública'
}

df_oferta = df_denue_raw[df_denue_raw['codigo_act'].isin(codigos_cuidado.keys())].copy()
df_oferta['TIPO_SERVICIO'] = df_oferta['codigo_act'].map(codigos_cuidado)
df_oferta['SECTOR'] = np.where(df_oferta['codigo_act'].str.endswith('2'), 'Público', 'Privado')

# Crear llaves geográficas compatibles con el json de alcaldías (no se utiliza en este código)
df_oferta['cve_ent'] = df_oferta['cve_ent'].astype(str).str.zfill(2)
df_oferta['cve_mun'] = df_oferta['cve_mun'].astype(str).str.zfill(3)
df_oferta['CVEGEO'] = df_oferta['cve_ent'] + df_oferta['cve_mun']

print(f"\nTotal de unidades de cuidado encontradas en CDMX: {len(df_oferta)}")
print(df_oferta['TIPO_SERVICIO'].value_counts())

Columnas del CSV del Censo: ['ENTIDAD', 'NOM_ENT', 'MUN', 'NOM_MUN', 'LOC', 'NOM_LOC', 'LONGITUD', 'LATITUD', 'ALTITUD', 'POBTOT', 'POBFEM', 'POBMAS']

Nivel detectado: LOCALIDAD/ITER | Filas limpias: 649
  CVEGEO                               NOM_MUN  POBTOT  DEMANDA_CUIDADO_TOTAL  \
1  09000  Total de la entidad Ciudad de México     320                     89   
2  09000  Total de la entidad Ciudad de México     364                     86   
4  09002                          Azcapotzalco  432205                 128625   
6  09003                              Coyoacán  614447                 193890   
8  09004                 Cuajimalpa de Morelos  186693                  43953   

   PE_INAC_F  
1         51  
2         59  
4      89796  
6     130022  
8      35010  
------------------------------------------------------------
Primeras 10 columnas del DENUE: ['id', 'clee', 'nom_estab', 'raz_social', 'codigo_act', 'nombre_act', 'per_ocu', 'tipo_vial', 'nom_vial', 'tipo_v_e_1']

Tota

In [5]:
# ===========================================================================================
#     Tabla para ver índice de prioridad, niños por guardería, mayores por centor, etc.
# ===========================================================================================

# Filtrar por alcaldías
df_alcaldias = df_censo_raw[df_censo_raw['NOM_LOC'].str.strip() == 'Total del Municipio'].copy()

# Asegurar que las columnas numéricas estén limpias en este subconjunto
cols_censo = ['POBTOT', 'POBFEM', 'P_0A2', 'P_3A5', 'P_60YMAS', 'PCON_DISC', 'PEA_F', 'PE_INAC_F', 'HOGJEF_F', 'TOTHOG']
for col in cols_censo:
    if col in df_alcaldias.columns:
        df_alcaldias[col] = pd.to_numeric(
            df_alcaldias[col].astype(str).str.replace('*', '0', regex=False).str.replace('N/D', '0', regex=False),
            errors='coerce'
        ).fillna(0)

# Llave geográfica
df_alcaldias['CVEGEO'] = df_alcaldias['ENTIDAD'].astype(str).str.zfill(2) + df_alcaldias['MUN'].astype(str).str.zfill(3)

# Variables de Demanda desagregadas:
df_alcaldias['DEMANDA_INFANTIL_0A5'] = df_alcaldias['P_0A2'] + df_alcaldias['P_3A5']
df_alcaldias['DEMANDA_MAYORES_Y_DISC'] = df_alcaldias['P_60YMAS'] + df_alcaldias['PCON_DISC']
df_alcaldias['DEMANDA_TOTAL'] = df_alcaldias['DEMANDA_INFANTIL_0A5'] + df_alcaldias['DEMANDA_MAYORES_Y_DISC']

# Indicadores de género y vulnerabilidad (enfoque interseccional):
# Porcentaje de mujeres en edad de trabajar que están fuera del mercado laboral (PE_INAC_F)
df_alcaldias['PCT_MUJERES_INACTIVAS'] = (df_alcaldias['PE_INAC_F'] / (df_alcaldias['PEA_F'] + df_alcaldias['PE_INAC_F'])) * 100

# No viene el número exacto de personal por centro, estimar uno por etiqueta
mapa_personal = {
    '0 a 5 personas': 3,
    '6 a 10 personas': 8,
    '11 a 30 personas': 20,
    '31 a 50 personas': 40,
    '51 a 100 personas': 75,
    '101 a 250 personas': 150,
    '251 y más personas': 300
}
df_oferta['PERSONAL_EST'] = df_oferta['per_ocu'].str.strip().map(mapa_personal).fillna(5)

# Clasificar la oferta por grupo de población objetivo y sector
df_oferta['GRUPO_OBJETIVO'] = np.where(
    df_oferta['codigo_act'].isin(['624411', '624412', '611111', '611112']),
    'Infantil',
    'Mayores_Discapacidad'
)

# Agrupar por alcaldía
oferta_resumen = df_oferta.groupby('CVEGEO').agg(
    TOTAL_UNIDADES=('id', 'count'),
    UNIDADES_PUBLICAS=('SECTOR', lambda x: (x == 'Público').sum()),
    UNIDADES_PRIVADAS=('SECTOR', lambda x: (x == 'Privado').sum()),
    GUARDERIAS_PUB=('codigo_act', lambda x: (x == '624412').sum()),
    GUARDERIAS_PRIV=('codigo_act', lambda x: (x == '624411').sum()),
    PREESCOLAR_PUB=('codigo_act', lambda x: (x == '611112').sum()),
    MAYORES_DISC_PUB=('codigo_act', lambda x: x.isin(['623312', '624122']).sum()),
    MAYORES_DISC_PRIV=('codigo_act', lambda x: x.isin(['623311', '624121']).sum()),
    PERSONAL_PUB_TOTAL=('PERSONAL_EST', lambda x: x[df_oferta.loc[x.index, 'SECTOR'] == 'Público'].sum())
).reset_index()

# Cruzar Demanda (Censo) + Oferta (DENUE)
df_maestro = pd.merge(
    df_alcaldias[['CVEGEO', 'NOM_MUN', 'POBTOT', 'POBFEM', 'P_0A2', 'P_3A5', 'DEMANDA_INFANTIL_0A5',
                  'P_60YMAS', 'PCON_DISC', 'DEMANDA_MAYORES_Y_DISC', 'DEMANDA_TOTAL',
                  'PEA_F', 'PE_INAC_F', 'PCT_MUJERES_INACTIVAS', 'HOGJEF_F']],
    oferta_resumen,
    on='CVEGEO',
    how='left'
).fillna(0)

# Calcular Métricas de Brecha ("Desiertos de Cuidado")
# Niños de 0 a 2 años por cada Guardería Pública (Primera infancia estricta)
df_maestro['NINOS_0A2_POR_GUARDERIA_PUB'] = np.round(df_maestro['P_0A2'] / df_maestro['GUARDERIAS_PUB'].replace(0, 1), 1)

# Niños de 0 a 5 años por centro infantil público (Guardería + Preescolar)
df_maestro['INFANTES_POR_CENTRO_PUB'] = np.round(
    df_maestro['DEMANDA_INFANTIL_0A5'] / (df_maestro['GUARDERIAS_PUB'] + df_maestro['PREESCOLAR_PUB']).replace(0, 1), 1
)

# Adultos mayores y personas con discapacidad por cada centro público de atención
df_maestro['MAYORES_POR_CENTRO_PUB'] = np.round(
    df_maestro['DEMANDA_MAYORES_Y_DISC'] / df_maestro['MAYORES_DISC_PUB'].replace(0, 1), 1
)

# Proporción de Oferta Privada vs Total (Nivel de privatización del cuidado en la alcaldía)
df_maestro['PCT_OFERTA_PRIVADA'] = np.round((df_maestro['UNIDADES_PRIVADAS'] / df_maestro['TOTAL_UNIDADES']) * 100, 1)

# ÍNDICE DE DESIERTO DE CUIDADO PÚBLICO (0 a 100, donde 100 es la mayor urgencia de intervención)
# Normalizamos Min-Max: 40% Déficit Infantil Público + 30% Déficit Mayores Público + 30% % Mujeres Inactivas
def norm_minmax(serie):
    return (serie - serie.min()) / (serie.max() - serie.min()) * 100

df_maestro['INDICE_PRIORIDAD_CUIDADO'] = np.round(
    0.40 * norm_minmax(df_maestro['NINOS_0A2_POR_GUARDERIA_PUB']) +
    0.30 * norm_minmax(df_maestro['MAYORES_POR_CENTRO_PUB']) +
    0.30 * norm_minmax(df_maestro['PCT_MUJERES_INACTIVAS']),
    1
)

# Ordenar de mayor a menor urgencia
df_maestro = df_maestro.sort_values(by='INDICE_PRIORIDAD_CUIDADO', ascending=False).reset_index(drop=True)

print(f"Total de alcaldías procesadas: {len(df_maestro)}\n")
cols_mostrar = [
    'NOM_MUN', 'INDICE_PRIORIDAD_CUIDADO', 'NINOS_0A2_POR_GUARDERIA_PUB',
    'MAYORES_POR_CENTRO_PUB', 'PCT_MUJERES_INACTIVAS', 'PCT_OFERTA_PRIVADA'
]
print(df_maestro[cols_mostrar].to_string(index=False))

Total de alcaldías procesadas: 16

               NOM_MUN  INDICE_PRIORIDAD_CUIDADO  NINOS_0A2_POR_GUARDERIA_PUB  MAYORES_POR_CENTRO_PUB  PCT_MUJERES_INACTIVAS  PCT_OFERTA_PRIVADA
            Xochimilco                      68.1                       3437.2                  7645.6              46.448042                37.5
            Iztapalapa                      67.7                       3047.7                 16658.5              46.457464                47.8
     Gustavo A. Madero                      53.0                       1602.8                 17011.3              47.966815                45.9
               Tlalpan                      52.8                       1047.8                 48127.3              44.319893                59.2
          Azcapotzalco                      47.3                       1071.3                 34216.7              44.869508                38.4
 Cuajimalpa de Morelos                      45.6                       2007.5                 1

In [6]:
import numpy as np

# ===========================================================================================
#             Simulador de solución (prueba para luego agregar a observable)
# ===========================================================================================

# Parámetros
# Se simula la apertura de NUEVAS_GUARDERIAS número de guarderías
# y de NUEVOS_CENTROS_DIA número de centros de cuidado
NUEVAS_GUARDERIAS = 50
NUEVOS_CENTROS_DIA = 30

# Filtrar las alcaldías prioritarias (Top 8 con Índice >= mediana)
top_prioridad = df_maestro['INDICE_PRIORIDAD_CUIDADO'] >= df_maestro['INDICE_PRIORIDAD_CUIDADO'].median()

# Calcular pesos proporcionales según volumen de demanda y severidad del déficit
peso_infantil = np.where(top_prioridad, df_maestro['P_0A2'] * df_maestro['NINOS_0A2_POR_GUARDERIA_PUB'], 0)
peso_mayores = np.where(top_prioridad, df_maestro['DEMANDA_MAYORES_Y_DISC'] * df_maestro['MAYORES_POR_CENTRO_PUB'], 0)

# Asignar las nuevas unidades matemáticamente
df_maestro['NUEVAS_GUARDERIAS_ASIGNADAS'] = np.round((peso_infantil / peso_infantil.sum()) * NUEVAS_GUARDERIAS).astype(int)
df_maestro['NUEVOS_CENTROS_MAYORES_ASIGNADOS'] = np.round((peso_mayores / peso_mayores.sum()) * NUEVOS_CENTROS_DIA).astype(int)

# Recalcular indicadores de brecha DESPUÉS de la intervención
df_maestro['GUARDERIAS_PUB_POST'] = df_maestro['GUARDERIAS_PUB'] + df_maestro['NUEVAS_GUARDERIAS_ASIGNADAS']
df_maestro['NINOS_0A2_POR_GUARDERIA_POST'] = np.round(
    df_maestro['P_0A2'] / df_maestro['GUARDERIAS_PUB_POST'].replace(0, 1), 1
)
df_maestro['REDUCCION_BRECHA_INFANTIL_PCT'] = np.round(
    (1 - (df_maestro['NINOS_0A2_POR_GUARDERIA_POST'] / df_maestro['NINOS_0A2_POR_GUARDERIA_PUB'])) * 100, 1
)

# Estimar el impacto directo en mujeres cuidadoras y horas liberadas (promedio 28 hrs/semana según ENUT)
df_maestro['MUJERES_CUIDADORAS_BENEFICIADAS'] = (
    (df_maestro['NUEVAS_GUARDERIAS_ASIGNADAS'] * 120) +
    (df_maestro['NUEVOS_CENTROS_MAYORES_ASIGNADOS'] * 150)
)
df_maestro['HORAS_SEMANALES_LIBERADAS'] = df_maestro['MUJERES_CUIDADORAS_BENEFICIADAS'] * 28

# Imprimir tabla de resultados del simulador
print("=== RESULTADO DEL SIMULADOR DE POLÍTICA PÚBLICA (TOP 8 ALCALDÍAS PRIORITARIAS) ===")
cols_sim = [
    'NOM_MUN',
    'INDICE_PRIORIDAD_CUIDADO',
    'NUEVAS_GUARDERIAS_ASIGNADAS',
    'NUEVOS_CENTROS_MAYORES_ASIGNADOS',
    'NINOS_0A2_POR_GUARDERIA_PUB',
    'NINOS_0A2_POR_GUARDERIA_POST',
    'REDUCCION_BRECHA_INFANTIL_PCT',
    'MUJERES_CUIDADORAS_BENEFICIADAS'
]
df_resultado_sim = df_maestro[df_maestro['MUJERES_CUIDADORAS_BENEFICIADAS'] > 0][cols_sim]
print(df_resultado_sim.to_string(index=False))

print("\n--- RESUMEN EJECUTIVO DE IMPACTO EN LA CDMX ---")
print(f"Total nuevas guarderías asignadas: {df_maestro['NUEVAS_GUARDERIAS_ASIGNADAS'].sum()}")
print(f"Total nuevos centros de día asignados: {df_maestro['NUEVOS_CENTROS_MAYORES_ASIGNADOS'].sum()}")
print(f"Total mujeres cuidadoras beneficiadas: {df_maestro['MUJERES_CUIDADORAS_BENEFICIADAS'].sum():,}")
print(f"Total horas de cuidado no remunerado liberadas por semana: {df_maestro['HORAS_SEMANALES_LIBERADAS'].sum():,} hrs")

=== RESULTADO DEL SIMULADOR DE POLÍTICA PÚBLICA (TOP 8 ALCALDÍAS PRIORITARIAS) ===
              NOM_MUN  INDICE_PRIORIDAD_CUIDADO  NUEVAS_GUARDERIAS_ASIGNADAS  NUEVOS_CENTROS_MAYORES_ASIGNADOS  NINOS_0A2_POR_GUARDERIA_PUB  NINOS_0A2_POR_GUARDERIA_POST  REDUCCION_BRECHA_INFANTIL_PCT  MUJERES_CUIDADORAS_BENEFICIADAS
           Xochimilco                      68.1                            7                                 1                       3437.2                        1249.9                           63.6                              990
           Iztapalapa                      67.7                           25                                 8                       3047.7                        1316.1                           56.8                             4200
    Gustavo A. Madero                      53.0                            8                                 6                       1602.8                        1175.4                           26.7               

In [7]:
from google.colab import files

# ==========================================
# EXPORTACIÓN LIMPIA Y FORMATEADA PARA OBSERVABLE FRAMEWORK
# ==========================================

# Guardar alcaldias_maestro.json bien formateado y sin NaNs
df_maestro.fillna(0).to_json(
    'alcaldias_maestro.json',
    orient='records',
    force_ascii=False,
    indent=2
)

# Limpiar unidades del DENUE antes de exportar
df_puntos_export = df_oferta[~df_oferta['TIPO_SERVICIO'].str.contains('Preescolar')].copy()
df_puntos_export = df_puntos_export.dropna(subset=['latitud', 'longitud'])

# Si 'nom_estab' es NaN, usar 'raz_social'; si también es NaN, poner un texto descriptivo limpio
df_puntos_export['nom_estab'] = (
    df_puntos_export['nom_estab']
    .fillna(df_puntos_export['raz_social'])
    .fillna('Establecimiento de Cuidado (Sin nombre comercial)')
    .astype(str)
    .str.strip()
)

# Limpiar cualquier otro posible NaN en columnas de texto
cols_puntos = ['nom_estab', 'TIPO_SERVICIO', 'SECTOR', 'GRUPO_OBJETIVO', 'latitud', 'longitud', 'CVEGEO', 'per_ocu']
df_puntos_export = df_puntos_export[cols_puntos].fillna('No especificado')

# Exportar con indent=2 y estándar JSON estricto (sin NaN)
df_puntos_export.to_json(
    'unidades_denue.json',
    orient='records',
    force_ascii=False,
    indent=2
)

print(f"Unidades exportadas limpias: {len(df_puntos_export)}")
print("Verificación de nulos en nom_estab:", df_puntos_export['nom_estab'].isna().sum())

# Descargar los archivos corregidos
files.download('alcaldias_maestro.json')
files.download('unidades_denue.json')

Unidades exportadas limpias: 838
Verificación de nulos en nom_estab: 0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>